# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We will walk through the process of loading dataset metadata, reviewing its structure, extracting records, and conducting exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This helps guide extraction and analysis using the unique identifiers from the schema.

In [ ]:
# List all record sets and their fields by @id

recordsets = dataset.record_sets

print("Available record sets and their fields:\n")
record_set_ids = []
for idx, record_set in enumerate(recordsets):
    print(f"[{idx}] Record set name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    # List all fields
    if record_set.fields:
        print(f"    Fields:")
        for field in record_set.fields:
            print(f"        - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

# For demonstration, extract the @id of the first record set
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None
print('Record sets:', record_set_ids)
print('Selecting first record set @id:', selected_record_set_id)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview section.

In [ ]:
# Extract data from all available record sets

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  No records found for this record set.")
    except Exception as e:
        print(f"  Could not load records due to error: {e}")
print()    
# For demo, show first 5 rows of the DataFrame from the first record set
if selected_record_set_id in dataframes:
    print(f"First 5 rows of record set: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to prepare the data for further analysis:
- Filtering by numeric criteria
- Normalization of fields
- Grouping/categorization

In [ ]:
# Example: Filter, normalize, and group a numeric field
#
# Please replace these @id values with the correct values based on the overview above for your workflow.

import numpy as np

record_set_id = selected_record_set_id
df = dataframes[record_set_id] if record_set_id in dataframes else None

if df is not None and not df.empty:
    # Try to select a numeric field (find one by dtype if possible)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Default to first numeric column
        print(f"Selected numeric field: {numeric_field_id}")
        threshold = float(df[numeric_field_id].mean())
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())
    else:
        print("No numeric fields found for EDA.")

    # Example: Group by a categorical field if present
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    group_field_id = None
    for col in categorical_cols:
        # Select first column with fewer than 10 unique values as group field
        if df[col].nunique() > 1 and df[col].nunique() <= 10:
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped = df.groupby(group_field_id).mean(numeric_only=True)
        display(grouped)
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions using built-in plotting tools.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty:
    if df.select_dtypes(include=[np.number]).shape[1] >= 1:
        num_field = df.select_dtypes(include=[np.number]).columns[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[num_field], bins=30, kde=True)
        plt.title(f"Distribution of {num_field}")
        plt.xlabel(num_field)
        plt.ylabel("Frequency")
        plt.show()

    if df.select_dtypes(include='object').shape[1] >= 1 and df.select_dtypes(include=[np.number]).shape[1] >= 1:
        cat_field = df.select_dtypes(include='object').columns[0]
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=cat_field, y=num_field, data=df)
        plt.title(f"{num_field} by {cat_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, overview, extract, and analyze data from the FAIR² mass spectrometry dataset using `mlcroissant`.

- We loaded the full Croissant schema and reviewed available record sets and fields by their `@id`.
- We extracted records, automatically detected numeric/categorical fields, performed filtering and normalization, and produced simple visualizations.
- All schema elements were referenced by their `@id`, ensuring clarity and reproducibility for FAIR workflows.

You can now use this workflow to perform additional domain-specific analysis or pipeline steps tailored to your research questions.